# 🤖 Step 4 of RAG — Putting It All Together with LangChain

In the previous notebooks we:

1. **Read, chunked & embedded** our document (`2 - Read Chunk and Embed`)
2. **Stored** those embeddings in PostgreSQL via pgvector (`3 - Store Embeddings`)
3. **Queried** the database with all three retrieval methods (`4 - Vector Database Retrieval`)

Now we add the final piece — the **LLM** — using **LangChain** to wire everything together:

```
User Question
     │
     ▼
  Embed Query  ──►  Ensemble Retrieval (Dense + Sparse + ColBERT)
                              │
                              ▼
                    Rank & deduplicate  (RRF)
                              │
                              ▼
                    LangChain RAG Chain
                    (prompt | llm | parser)
                              │
                              ▼
                         Answer ✅
```

> 💡 **Why LangChain?** LangChain's **LangChain Expression Language (LCEL)** lets us compose retrieval, prompting, and generation into a clean, readable pipeline using the `|` operator — exactly like in `langchain_chain_demo.py`.

## 0. Setup & Imports

In [ ]:
import os
import json
import numpy as np
import psycopg2
from pathlib import Path
from dotenv import load_dotenv
from munch import Munch

# LangChain
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Embedding model
from FlagEmbedding import BGEM3FlagModel

### Load config & secrets

In [ ]:
# Load config
config_path = (Path.cwd() / "../config/config.yaml").resolve()
with open(config_path, "r") as f:
    config = Munch.fromYAML(f)

# Load secrets from .env
load_dotenv((Path.cwd() / "../.env").resolve())
SUPABASE_PASSWORD = os.getenv("SUPABASE_PASSWORD")
GROQ_API_KEY      = os.getenv("GROQ_API_KEY")

if not SUPABASE_PASSWORD:
    raise ValueError("SUPABASE_PASSWORD not found in .env")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env")

print("✅ Config and secrets loaded")

### Connect to the database and load models

In [ ]:
# PostgreSQL connection helper
DATABASE_URL = (
    f"postgresql://{config.supabase.user}:{SUPABASE_PASSWORD}@"
    f"{config.supabase.host}:{config.supabase.port}/{config.supabase.database}"
)

def get_connection():
    return psycopg2.connect(DATABASE_URL)

sql_dir = (Path.cwd() / "sql").resolve()

print("✅ Database connection ready")

In [ ]:
# BGE-M3 embedding model
print(f"Loading {config.embedding.model} …")
embed_model = BGEM3FlagModel(config.embedding.model, use_fp16=True)
print("✅ Embedding model loaded")

---

## 1. Ensemble Retrieval

We reuse the three retrieval functions from Notebook 4 and combine them with **Reciprocal Rank Fusion (RRF)**.

RRF merges ranked lists from different retrievers without needing their scores to be on the same scale:

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k = 60$ is a smoothing constant. A chunk that ranks highly across *multiple* methods scores highest overall.

In [ ]:
def retrieve_dense(q_dense, top_k):
    sql = (sql_dir / "09_query_dense.sql").read_text()
    vec_str = f"[{','.join(map(str, q_dense.tolist()))}]"
    conn = get_connection()
    cur  = conn.cursor()
    try:
        cur.execute(sql, (vec_str, vec_str, top_k))
        return [{"chunk_id": r[0], "chunk_text": r[1], "score": float(r[2])} for r in cur.fetchall()]
    finally:
        cur.close(); conn.close()


def retrieve_sparse(q_sparse, top_k):
    sql = (sql_dir / "10_query_sparse.sql").read_text()
    sparse_json = json.dumps({str(k): float(v) for k, v in q_sparse.items()})
    conn = get_connection()
    cur  = conn.cursor()
    try:
        cur.execute(sql, (sparse_json, top_k))
        return [{"chunk_id": r[0], "chunk_text": r[1], "score": float(r[2])} for r in cur.fetchall()]
    finally:
        cur.close(); conn.close()


def retrieve_colbert(q_colbert, top_k):
    sql = (sql_dir / "11_query_colbert.sql").read_text()
    candidate_pool = top_k * 3
    chunk_scores = {}
    conn = get_connection()
    cur  = conn.cursor()
    try:
        for token_vec in q_colbert:
            vec_str = f"[{','.join(map(str, token_vec.tolist()))}]"
            cur.execute(sql, (vec_str, candidate_pool))
            for chunk_id, chunk_text, max_sim in cur.fetchall():
                entry = chunk_scores.setdefault(chunk_id, {"chunk_text": chunk_text, "sims": []})
                entry["sims"].append(float(max_sim))
    finally:
        cur.close(); conn.close()
    results = [
        {"chunk_id": cid, "chunk_text": d["chunk_text"], "score": float(np.mean(d["sims"]))}
        for cid, d in chunk_scores.items()
    ]
    return sorted(results, key=lambda x: x["score"], reverse=True)[:top_k]


def ensemble_retrieve(query: str) -> str:
    """
    Embed the query, run all three retrievers, fuse with RRF,
    and return the top chunks as a single formatted context string.
    """
    top_k = config.retrieval.top_k

    # 1. Embed the query with all three approaches
    q_out = embed_model.encode(
        [query],
        max_length=config.embedding.max_length,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=True,
    )

    # 2. Run all three retrievers
    all_results = [
        retrieve_dense(q_out["dense_vecs"][0], top_k),
        retrieve_sparse(q_out["lexical_weights"][0], top_k),
        retrieve_colbert(q_out["colbert_vecs"][0], top_k),
    ]

    # 3. Reciprocal Rank Fusion (k=60)
    rrf_scores = {}
    chunk_texts = {}
    for results in all_results:
        for rank, item in enumerate(results, start=1):
            cid = item["chunk_id"]
            rrf_scores[cid]  = rrf_scores.get(cid, 0.0) + 1.0 / (60 + rank)
            chunk_texts[cid] = item["chunk_text"]

    # 4. Sort by fused score and format as a context string
    top_chunks = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return "\n\n---\n\n".join(
        f"[Chunk {i+1}]\n{chunk_texts[cid]}"
        for i, (cid, _) in enumerate(top_chunks)
    )


print("✅ Ensemble retrieval function defined")

---

## 2. Build the LangChain RAG Chain

LangChain's **LCEL** (`|` operator) lets us compose steps declaratively. Our chain has three stages:

```
{ context: ensemble_retrieve(question), question: passthrough }
          │
          ▼
     ChatPromptTemplate
          │
          ▼
       ChatGroq  (LLaMA-3.1 via Groq)
          │
          ▼
     StrOutputParser
```

This is the same pattern used in `langchain_chain_demo.py` — just with ensemble retrieval replacing the first chain step.

In [ ]:
# 🤖 LLM — Groq via LangChain
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=config.groq.model,
    temperature=config.groq.temperature,
    max_tokens=config.groq.max_tokens,
)

# 📝 Prompt template
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant that answers questions using only the provided context.\n"
        "If the answer is not in the context, say \"I don't know based on the provided context.\"\n"
        "Be concise and accurate.",
    ),
    (
        "user",
        "Context:\n{context}\n\nQuestion: {question}",
    ),
])

# 🔗 RAG chain (LCEL)
rag_chain = (
    {
        "context":  RunnableLambda(lambda x: ensemble_retrieve(x["question"])),
        "question": RunnablePassthrough() | RunnableLambda(lambda x: x["question"]),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ LangChain RAG chain built")

---

## 3. Ask a Question!

Invoking the chain is a single call. LangChain handles the rest — retrieval, prompt formatting, LLM call, and output parsing.

In [ ]:
QUESTION = "What did Alice find at the bottom of the rabbit hole?"

answer = rag_chain.invoke({"question": QUESTION})

print(f"❓ {QUESTION}\n")
print(f"🤖 {answer}")

---

## 4. Interactive Mode

Change `QUESTION` below and re-run the cell to ask anything about the book.

In [ ]:
QUESTION = "How does Alice change size throughout the story?"
# QUESTION = "Who is the Mad Hatter and why is he mad?"
# QUESTION = "What happens at the Queen's croquet game?"

answer = rag_chain.invoke({"question": QUESTION})

print(f"❓ {QUESTION}\n")
print(f"🤖 {answer}")

---

## 5. Bonus — Streaming Responses

One of LangChain's built-in benefits is **streaming**: instead of waiting for the full response, tokens arrive as they are generated. This is a one-line change from `.invoke()` to `.stream()`.

In [ ]:
QUESTION = "Describe the Mad Tea-Party scene."

print(f"❓ {QUESTION}\n")
print("🤖 ", end="", flush=True)

for chunk in rag_chain.stream({"question": QUESTION}):
    print(chunk, end="", flush=True)

print()  # newline at the end

---

## 6. Summary

Here is the complete RAG pipeline we built across all five notebooks:

| Step | Notebook | What it does |
|---|---|---|
| **Chunk** | `2 - Read Chunk and Embed` | Split the document into manageable pieces |
| **Embed** | `2 - Read Chunk and Embed` | Convert chunks to dense, sparse & ColBERT vectors |
| **Store** | `3 - Store Embeddings` | Persist vectors in PostgreSQL with pgvector |
| **Retrieve** | `4 - Vector DB Retrieval` | Query the DB with dense, sparse & ColBERT |
| **Fuse** | `5 - RAG` *(this notebook)* | Combine results with Reciprocal Rank Fusion |
| **Generate** | `5 - RAG` *(this notebook)* | LangChain LCEL chain → Groq / LLaMA-3.1 → answer |

### Key takeaways

- **RAG = Retrieval + Generation**. The LLM only sees what the retriever gives it — retrieval quality is the bottleneck.
- **Ensemble retrieval + RRF** combines the strengths of dense (semantic), sparse (keyword), and ColBERT (precise token-level) methods.
- **LangChain LCEL** (`|`) makes it easy to compose retrieval, prompting, and generation into a single readable pipeline — and streaming comes for free.

### Further improvements to explore

- **Re-ranking**: add a cross-encoder step after retrieval to re-score the top chunks
- **Conversational memory**: use `ConversationBufferMemory` to support multi-turn Q&A
- **Query expansion**: rewrite or expand the user question before embedding to improve recall
- **Metadata filtering**: pre-filter by chapter or date before running vector search